# Lava Visualization and OpenEO

This notebook demonstrates how to visualize volcanic lava flows using Sentinel-2 imagery. The visualization applies cartographic-style rendering that the author can switch between different purposes.

The available styles include:
1. A continuous heatmap-like visualization where lava intensity (from cooling to active) is encoded using SWIR B12-based thresholds and blended into a base map that preserves soil, built-up areas, vegetation, and water bodies.
2. Binary lava detection styles using red or yellow coloring to indicate the presence or absence of active lava.
3. Context-preserving binary styles that highlight lava pixels while maintaining the underlying land-cover information for situational awareness.

The visualization uses SWIR Band 12 (2202.4 nm) which is highly sensitive to thermal emission from molten lava. Active lava flows produce extremely high radiance in this band, making it ideal for detecting and mapping volcanic activity. The visualization also includes optional cloud avoidance logic to reduce false detections.

In [ ]:
import openeo
import matplotlib.pyplot as plt
from PIL import Image
from openeo.processes import array_create, if_, and_, array_element
from openeo.api.process import Parameter

# Connect to OpenEO Backend
Connect to the OpenEO backend and authenticate using OpenID Connect.

In [ ]:
connection = openeo.connect(
    url="https://api.explorer.eopf.copernicus.eu/openeo"
    # url="http://127.0.0.1:8081/"
).authenticate_oidc_authorization_code()

# Define Area of Interest
Define the spatial extent for our analysis. This example uses coordinates for Battleship Mountain, B.C., Canada

In [ ]:
# Northern Sardinia, Italy
# spatial_extent = {
#     "west": 8.5,
#     "east": 9.5,
#     "north": 41.5,
#     "south": 40.5
# }
# Cumbre Vieja, La Palma, Spain
spatial_extent = {
    "west": -17.94,
    "east": -17.86,
    "north": 28.64,
    "south": 28.58
}

collection_id = "sentinel-2-l2a"
bounding_box = Parameter(
    "bounding_box",
    description="Bounding box for the area of interest",
    default={
        "west": spatial_extent["west"],
        "east": spatial_extent["east"],
        "north": spatial_extent["north"],
        "south": spatial_extent["south"],
    },
)
time = Parameter(
    "time",
    description="Time range for the data",
    default=["2021-09-19", "2022-01-10"],
)
bands = Parameter(
    "bands",
    description="Bands to load",
    default=["reflectance|b02", "reflectance|b03", "reflectance|b04", "reflectance|b08", "reflectance|b11", "reflectance|b12"]
)
cloud_cover = Parameter(
    "cloud_cover",
    description="Maximum cloud cover percentage",
    default=70,
)

# Load Sentinel-2 Data
Load Sentinel-2 L2A data. We need multiple bands to visualize the underlying land-cover information, water, and wildfire.

- **B02**: Blue, 492.4 nm
- **B03**: Green, 559.8 nm
- **B04**: Red, 664.6 nm
- **B08**: NIR, 832.8 nm
- **B8A**: Narrow NIR, 864.7 nm
- **B11**: SWIR, 1613.7 nm
- **B12**: SWIR, 2202.4 nm


In [ ]:
s2cube = connection.load_collection(
    collection_id=collection_id,
    spatial_extent=bounding_box,
    temporal_extent=time,
    bands=bands,
    properties={
        "eo:cloud_cover": lambda x: x <= cloud_cover,
    },
)
s2cube = s2cube.reduce_dimension(dimension="time", reducer="first")  # single timestamp for PNG


### Lava Visualization Styles

The `lava_viz_style` helper lets you choose among four preset visual styles for highlighting active lava pixels. You need to enable hotspot rendering to use any of these styles (`hotspot = 1`):

- **`style = 1` – Continuous heatmap**: Shows lava with a smooth color ramp, where **active molten lava** appears in bright yellow and **cooling lava** in darker reds.
- **`style = 2` – Binary red mask**: Highlights pixels classified as lava in **bright red**, with non-lava areas shown in a neutral background.
- **`style = 3` – Binary yellow mask**: Similar to style 2, but lava pixels are shown in **bright yellow**, useful on darker backgrounds or when combining with other layers.
- **`style = 4` – Contextual lava map**: Emphasizes lava pixels with a bright color while also rendering other land-cover types (e.g. urban areas, vegetation) with more subdued tones, giving context to where the lava flow occurs.

The lava layer is drawn on top of a base map that shows land cover such as soil, built-up areas, vegetation, and water. In this notebook, the default base map is a blend of urban composite, true color, and color-corrected true color.

In [ ]:
# If user wants to visualize the lava hotspots, they can set the hotspot variable to 1
hotspot = 1

# If user wants to visualize the lava in a continuous heatmap-style, they can set the style variable to 1
style = 1

# Define the file name for downloading the image
if hotspot:
    if style:
        file_name = f"lava_hotspot_style{style}.png"

# Threshold for lava intensity using B12 (higher thresholds than wildfire due to higher thermal emission)
# Lava typically shows very high B12 values (often saturated)
lavaThreshold = [0.8, 0.5, 0.3, 0.2]  # B12 reflectance thresholds
lavaSensitivity = 1.0
boost = 1

# Threshold for cloud probability
cloudAvoidance = 1
cloudAvoidanceThreshold = 245
avoidanceHelper = 0.8

# Image brightness, saturation, and contrast
offset = -0.000
saturation = 1.10
brightness = 1.00
sMin = 0.01
sMax = 0.99

# Threshold for water
waterHighlight = 0
waterBoost = 2.0
waterHelper = 0.2

# Index thresholds
NDVI_threshold = -0.15
NDWI_threshold = 0.15

In [ ]:
def lava_viz_style(base_arr, b02, b03, b04, b12):
    """
    Visualize lava flows using SWIR Band 12.
    B12 (2202.4 nm) is highly sensitive to thermal emission from molten lava.
    """
    base_r = array_element(base_arr, 0)
    base_g = array_element(base_arr, 1)
    base_b = array_element(base_arr, 2)

    if hotspot:
        # Use B12 directly for lava detection (strong thermal emission in SWIR)
        lava_intensity = b12

        if style == 1:
            # Continuous heatmap based on B12 intensity
            mask_0 = lava_intensity > (lavaThreshold[0] / lavaSensitivity)

            mask_1 = and_(
                lava_intensity > (lavaThreshold[1] / lavaSensitivity), not_(mask_0)
            )

            mask_2 = and_(
                lava_intensity > (lavaThreshold[2] / lavaSensitivity),
                not_(or_(mask_0, mask_1)),
            )

            mask_3 = and_(
                lava_intensity > (lavaThreshold[3] / lavaSensitivity),
                not_(or_(mask_0, or_(mask_1, mask_2))),
            )

            # Any lava present
            mask_any = or_(mask_0, or_(mask_1, or_(mask_2, mask_3)))

            # Red channel boosted for all lava
            r = base_r + (boost * 0.6 * b12 * mask_any)

            # Green channel weighted by lava intensity classes (brighter = more active)
            g = base_g + (
                boost
                * b12
                * (
                    0.5 * mask_0  # Active molten lava - bright yellow
                    + 0.3 * mask_1  # Hot lava - orange-yellow
                    + 0.15 * mask_2  # Cooling lava - orange
                    + 0.0 * mask_3  # Cool crust - red only
                )
            )

            # Blue channel remains unchanged
            b = base_b

            return array_create([r, g, b])

        # Binary lava detection using B12 threshold
        lava_mask = lava_intensity > (lavaThreshold[3] / lavaSensitivity)

        if style == 2:
            # Binary red mask
            return array_create(
                [
                    if_(lava_mask, 1, base_r),
                    if_(lava_mask, 0, base_g),
                    if_(lava_mask, 0, base_b),
                ]
            )

        if style == 3:
            # Binary yellow mask
            return array_create(
                [
                    if_(lava_mask, 1, base_r),
                    if_(lava_mask, 1, base_g),
                    if_(lava_mask, 0, base_b),
                ]
            )

        if style == 4:
            # Contextual lava map
            return array_create(
                [
                    if_(lava_mask, base_r + 0.3, base_r),
                    if_(lava_mask, base_g - 0.1, base_g),
                    if_(lava_mask, base_b - 0.2, base_b),
                ]
            )

    return base_arr

In [ ]:
def lava_visualization(data):
    # Extract bands
    B02, B03, B04, B08, B11, B12 = (
        data[0],
        data[1],
        data[2],
        data[3],
        data[4],
        data[5],
    )

    # Calculate indexes for vegetation and water
    NDWI = (B03 - B08) / (B03 + B08)
    NDVI = (B08 - B04) / (B08 + B04)

    # True colors with color correction
    naturalColorsCC = naturalColorsCC_arr(brightness, B04, B03, B02, offset)

    # True colors
    naturalColors = naturalColors_arr(brightness, B04, B03, B02, offset)

    # SWIR composite for urban areas
    urban = urban_arr(brightness, B12, B11, B04, offset)

    # Create lava visualization base map as an RGB combination
    baseViz = layer_blend(urban, naturalColors, naturalColorsCC, 10, 40, 50)

    # Coloring the water
    baseViz_red = array_element(baseViz, 0)

    water_detection = and_(and_(NDVI < NDVI_threshold, NDWI > NDWI_threshold), B04 < waterHelper)

    baseViz_green = if_(
        waterHighlight,
        if_(
            water_detection,
            array_element(baseViz, 1) * 1.2 * waterBoost + 0.1,
            array_element(baseViz, 1)
        ),
        array_element(baseViz, 1)
    )

    baseViz_blue = if_(
        waterHighlight,
        if_(
            water_detection,
            array_element(baseViz, 2) * 1.5 * waterBoost + 0.2,
            array_element(baseViz, 2)
        ),
        array_element(baseViz, 2)
    )

    baseViz = array_create([baseViz_red, baseViz_green, baseViz_blue])

    # Enhance saturation only (no stretch to preserve clouds and bright areas)
    baseViz = satEnh(baseViz, saturation)

    # Apply lava visualization style using B12
    baseViz = lava_viz_style(baseViz, B02, B03, B04, B12)

    return baseViz

## Save and Download the Image

Generate the final lava visualization image and download it as a PNG file.

In [ ]:
lava_viz_image = s2cube.apply_dimension(dimension="bands", process=lava_visualization)

lava_viz_image = lava_viz_image.linear_scale_range(input_min=0, input_max=1, output_min=0, output_max=255)

lava_viz_image = lava_viz_image.save_result("PNG")

print(file_name)

In [ ]:
lava_viz_image.download(file_name)

## Visualize the Result

Display the generated lava flow map with different styles.
1. A continuous heatmap-like visualization where lava intensity (from cooling crust to active molten lava) is encoded using B12 SWIR-based thresholds.
2. Binary lava detection styles using red or yellow coloring to indicate the presence or absence of active lava flows.

In [ ]:
# Load and display the image
img = Image.open("lava_hotspot_style1.png")

fig, ax = plt.subplots(figsize=(14, 10), dpi=100)
ax.imshow(img)
ax.set_title(
    "Lava Flow Visualization - Intensity Heatmap\n"
    "Yellow: Active Molten Lava (B12 > 0.8) | Orange-Yellow: Hot Lava (0.5 < B12 <= 0.8)\n"
    "Orange: Cooling Lava (0.3 < B12 <= 0.5) | Red: Cool Crust (0.2 < B12 <= 0.3)\n"
    "Cumbre Vieja, La Palma, Spain",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
bounding_box = Parameter(
    "bounding_box",
    description="Bounding box for the area of interest",
    default={
        "west": -5.0,
        "south": 40.0,
        "east": 5.0,
        "north": 50.0,
    },
)
time = Parameter(
    "time",
    description="Time range for the data",
    default=["2026-01-10", "2026-01-20"],
)
bands = Parameter(
    "bands",
    description="Bands to load",
    default=["B04", "B03", "B02"],
)
cloud_cover = Parameter(
    "cloud_cover",
    description="Maximum cloud cover percentage",
    default=20,
)

try:
    # Create XYZ service for web mapping
    service = connection.create_service(
        {
            "process_graph": result_png.flat_graph(),
            "parameters": [
                time.to_dict(),
                bounding_box.to_dict(),
                bands.to_dict(),
            ]
        },
        id=f"{metadata.get('algorithm_id', 'algorithm')}_service",
        type="XYZ",
        title=f"OpenEO UDP - {metadata.get('algorithm_name', 'Algorithm')}",
        description=f"{metadata.get('description', 'Algorithm visualization')}",
        configuration={
            "tile_size": 256,
            "minzoom": 5,
            "maxzoom": 14,
            "extent": [
                bounding_box.default["west"],
                bounding_box.default["south"],
                bounding_box.default["east"],
                bounding_box.default["north"],
            ],
            "scope": "public",
        },
    )
    
    print(f"XYZ Service created: {service.service_id}")
    service_id = service.service_id
    
except Exception as e:
    print(f"Service creation failed: {e}")
    service_id = None

### Technical Notes

**Why SWIR Band 12 for Lava Detection?**

SWIR Band 12 (2202.4 nm) is ideal for detecting molten lava because:
- Molten lava (700-1200°C) emits strong thermal radiation in the SWIR range
- B12 can detect thermal anomalies even when partially obscured by volcanic gases
- Active lava flows often saturate B12, making detection straightforward
- Cooling lava shows progressively lower B12 values, enabling intensity mapping

**Thresholds:**
- The default thresholds are calibrated for typical volcanic eruptions
- Adjust `lavaThreshold` values based on the specific volcano and eruption intensity
- Very active eruptions may require higher thresholds to differentiate intensity levels

## Attribution

This openEO User-Defined Process is adapted from wildfire visualization techniques for volcanic lava flow monitoring.

**Methodology:** Uses SWIR Band 12 thermal emission detection, similar to approaches used in:
- Coppola, D., et al. (2016). Enhanced volcanic hot-spot detection using MODIS IR data.
- Wright, R., et al. (2004). MODVOLC: near-real-time thermal monitoring of global volcanism.

The visualization leverages the strong thermal signature of molten lava in the SWIR bands to map active lava flows and their cooling patterns.